<a href="https://colab.research.google.com/github/Kaviyarasi-Sasiperumal/AI_-Based_-Document-_Search_-and-_Knowledge-_Retrieval_-with-_Conversational_Interface/blob/main/Milestone_4_Deployment_%26__Final_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf sentence-transformers faiss-cpu transformers gradio torch


In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import gradio as gr
import time


In [ ]:
documents = []
metadata = []
embedder = SentenceTransformer("all-MiniLM-L6-v2")
index = None
llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)


In [ ]:
def chunk_text(text, chunk_size=250, overlap=100):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i+chunk_size])
    return chunks


def load_document(file):
    global documents, metadata, index

    documents = []
    metadata = []

    reader = PdfReader(file.name)

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            chunks = chunk_text(text)
            for chunk in chunks:
                documents.append(chunk)
                metadata.append(f"{file.name} - page {page_num+1}")

    if not documents:
        return "❌ No readable text found in document."

    # Create embeddings & FAISS index
    embeddings = embedder.encode(documents)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))

    return f"✅ Document loaded successfully! Pages indexed: {len(metadata)}"


In [ ]:
!pip install gradio PyPDF2

In [ ]:
def chat_function(user_input, history):
    start_time = time.perf_counter()

    if user_input.lower() in ["hi", "hello", "hey"]:
        return "👋Hello! Ask Questions from Document."

    if not document_text:
        return "Please upload a document first."

    answer = get_answer_from_document(user_input)

    response_time = (time.perf_counter() - start_time) * 1000
    response_time = round(response_time, 2)

    return f"""📄 Answer:
{answer}

⏱ Response Time: {response_time} ms
"""


# -------- UI --------
with gr.Blocks() as demo:
    gr.Markdown("## 🤖  Document Chatbot ")

    with gr.Row():
        # ---- LEFT SIDE ----
        with gr.Column(scale=1):
            gr.Markdown("### 📂 Upload Document")
            file_input = gr.File(label="Upload PDF / TXT")
            status = gr.Textbox(label="Status", interactive=False)
            stats_box = gr.Textbox(label="Document Statistics", interactive=False)

            file_input.change(
                load_document,
                file_input,
                outputs=[status, stats_box]
            )

        # ---- RIGHT SIDE ----
        with gr.Column(scale=3):
            gr.ChatInterface(
                fn=chat_function,
                title="Ask questions"
            )

demo.launch()
